# Entraînement SAC : rebonds successifs

Choisir le kernel **Python (so101-robotic-learning)**, puis exécuter les cellules dans l'ordre.
L'environnement doit contenir le projet installé avec `pip install -e .` et MuJoCo.
Redémarrer le kernel après une modification des sources. EGL n'est pas imposé : sur un
serveur Linux sans écran, configurer le backend avant tout import MuJoCo si nécessaire.

La récompense vaut **5 × rebonds + 0.05 × shaping − 5 à la chute**. La réussite demande
5 rebonds consécutifs par défaut. Les checkpoints sont sélectionnés selon les métriques
physiques ; leurs scores ne sont pas directement comparables aux anciens entraînements.
Les logs sont écrits dans `runs/sac_bounce/`. Modèles, rapports et vidéos utilisent des sous-dossiers datés distincts.


In [ ]:
from datetime import datetime
from pathlib import Path
import json
import numpy as np
from IPython.display import JSON, Markdown, display

# Works from the repository root or any of its subdirectories.
root = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "pyproject.toml").is_file()
             and (path / "src/so101_robotic_learning").is_dir()), None)
if root is None:
    raise RuntimeError("Ouvrir ce notebook depuis le dépôt ou l'un de ses sous-dossiers.")
xml_path = root / "assets/mjcf/so101_new_calib copy.xml"
if not xml_path.is_file():
    raise FileNotFoundError(xml_path)

# MuJoCo selects the platform backend. Do not force EGL on Windows.
from so101_robotic_learning.bounce_rl.env.bounce_events import BounceConfig

from so101_robotic_learning.bounce_rl.rl import train_sac


## Paramètres

`reward`, si fourni à `train()`, désigne le shaping auxiliaire. Les paramètres d'environnement
passent par `env_kwargs`. Pour utiliser un historique, modifier `latent_steps` **avant** de
lancer l'entraînement et conserver exactement la même configuration lors du test du modèle.
PPO et SAC utilisent ici la même tâche et le même seuil de réussite.


In [ ]:
run_id = "sac_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
run_root = root

env_kwargs = {
    "camera_name": "gripper_cam",  # policy input only
    "render_profile": "train_fast",  # same profile for training and evaluation
    "video_render_profile": "standard",  # external camera only
    "video_camera_name": "overview_cam",  # visualization only
    "video_width": 320,
    "video_height": 240,
    "bounce_config": BounceConfig(success_bounces=5),
    "bounce_reward_weight": 5.0,
    "shaping_reward_weight": 0.05,
    "fall_penalty": 5.0,
    "max_episode_steps": 1024,
    "frame_skip": 20,
    "latent_steps": np.array([], dtype=np.int32),
}
train_config = {
    "xml_path": xml_path,
    "root": run_root,
    "model_dir_name": f"models/{run_id}",
    "env_kwargs": env_kwargs,
    "total_timesteps": 20_000,
    "nb_vec_env_train": 1,
    "nb_vec_env_test": 1,
    "learning_rate": 3e-4,
    "buffer_size": 3_000,
    "learning_starts": 1_000,
    "batch_size": 64,
    "eval_every_episodes": 10,
    "n_eval_episodes": 3,
    "eval_max_steps": env_kwargs["max_episode_steps"],
    "train_render_mode": None,
    "eval_render_mode": None,  # recording is enabled independently of the viewer
    "save_eval_video": True,
    "tensorboard_log": True,
    "run_subdir": "sac_bounce",
    "device": "auto",
}
RUN_TENSORBOARD = False  # optional inline dashboard
RUN_POLICY_TEST = False  # optional evaluation of the best checkpoint
print(f"Logs TensorBoard : {run_root / 'runs/sac_bounce'}")
print(f"Modèles et rapports : {run_root / train_config['model_dir_name']}")


## Entraînement

Cette cellule lance réellement 20 000 transitions par défaut. Adapter le budget avant exécution.
Les métriques TensorBoard sont écrites à chaque évaluation, après `eval_every_episodes` épisodes.


In [ ]:
train_sac.train(**train_config)


## Résultats de cet essai

Le taux de réussite, les séries de rebonds et la durée priment sur la récompense.
Un temps avant chute `null` signifie qu'aucune chute n'a été observée ; la dispersion
est définie seulement à partir de deux impacts valides dans un épisode.


In [ ]:
model_dir = run_root / train_config["model_dir_name"]
reports = [json.loads(path.read_text(encoding="utf-8"))
           for path in sorted(model_dir.glob("*_eval_*.json"))]
if reports:
    latest = max(reports, key=lambda report: report["training_timesteps"])
    best = max(reports, key=lambda report: report["selection_score"])
    display(JSON({
        "derniere_evaluation": latest["task_metrics"],
        "pas_entrainement": latest["training_timesteps"],
        "meilleur_score_de_tache": best["selection_score"],
        "checkpoint": str(model_dir / "sac_bounce_best.zip"),
    }))
else:
    print("Aucune évaluation : augmenter total_timesteps ou réduire eval_every_episodes.")


## TensorBoard

La commande ci-dessous utilise les logs de cet essai. Passer `RUN_TENSORBOARD=True`
pour ouvrir aussi le tableau dans le notebook. Les courbes sont regroupées sous `eval/`. Avec `save_eval_video=True`, les vidéos
d’évaluation sont intégrées en GIF dans **Images → eval/cameras**, sans export MP4.


In [ ]:
tensorboard_dir = run_root / "runs/sac_bounce"
print(f'tensorboard --logdir "{tensorboard_dir}"')
if RUN_TENSORBOARD:
    get_ipython().run_line_magic("load_ext", "tensorboard")
    get_ipython().run_line_magic("tensorboard", f'--logdir "{tensorboard_dir}"')


## Test facultatif du meilleur checkpoint

Passer `RUN_POLICY_TEST=True` dans la configuration. Le test garde les mêmes observations
et paramètres de tâche, écrit les métriques en JSON et enregistre les vidéos sans fenêtre native.


In [ ]:
if RUN_POLICY_TEST:
    train_sac.test(
        xml_path=xml_path,
        root=run_root,
        model_path=model_dir / "sac_bounce_best.zip",
        env_kwargs=env_kwargs,  # same observations, reward weights and success threshold
        render_mode="rgb_array_list",
        n_episodes=3,
        max_steps=env_kwargs["max_episode_steps"],
        save_test_video=True,
        video_dir_name=f"runs/sac_bounce/test_{run_id}",
    )
